# DBSCAN in Google Colab

This notebook demonstrates **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** with clear visualizations and practical parameter tuning.

## What you will learn
- How DBSCAN works
- The meaning of `eps` and `min_samples`
- How DBSCAN compares with K-Means
- How to choose a good `eps` using a k-distance plot

In [ ]:
# Install dependencies (safe to run in Colab)
!pip -q install scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

print("Libraries imported successfully.")

## 1. Create non-spherical dataset

DBSCAN is strong on non-spherical clusters where K-Means often struggles.

In [ ]:
X, _ = make_moons(n_samples=400, noise=0.08, random_state=42)
X = StandardScaler().fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=25, alpha=0.8, edgecolors="k", linewidth=0.3)
plt.title("Input Data (Two Moons)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 2. Compare K-Means vs DBSCAN

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X)

dbscan = DBSCAN(eps=0.25, min_samples=6)
db_labels = dbscan.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=km_labels, cmap="viridis", s=28, alpha=0.9, edgecolors="k", linewidth=0.2)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", s=220, marker="*", edgecolors="black")
axes[0].set_title("K-Means (K=2)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

core_mask = np.zeros_like(db_labels, dtype=bool)
core_mask[dbscan.core_sample_indices_] = True
noise_mask = db_labels == -1

axes[1].scatter(X[~noise_mask, 0], X[~noise_mask, 1], c=db_labels[~noise_mask], cmap="viridis", s=28, alpha=0.9, edgecolors="k", linewidth=0.2)
axes[1].scatter(X[noise_mask, 0], X[noise_mask, 1], c="red", marker="x", s=80, linewidth=2, label="Noise")
axes[1].set_title("DBSCAN (eps=0.25, min_samples=6)")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 3. Visualize DBSCAN point types

- **Core point**: has at least `min_samples` points inside its `eps`-neighborhood
- **Border point**: not core, but reachable from a core point
- **Noise point**: neither core nor border

In [ ]:
dbscan = DBSCAN(eps=0.25, min_samples=6)
labels = dbscan.fit_predict(X)

core_mask = np.zeros(len(X), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True
noise_mask = labels == -1
border_mask = (~core_mask) & (~noise_mask)

plt.figure(figsize=(7, 6))
plt.scatter(X[core_mask, 0], X[core_mask, 1], c="#2b8a3e", s=45, label="Core", edgecolors="k", linewidth=0.2)
plt.scatter(X[border_mask, 0], X[border_mask, 1], c="#f08c00", s=55, label="Border", edgecolors="k", linewidth=0.3)
plt.scatter(X[noise_mask, 0], X[noise_mask, 1], c="#c92a2a", s=70, marker="x", linewidth=1.8, label="Noise")
plt.title("DBSCAN Point Classification")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

## 4. Parameter sensitivity (`eps`, `min_samples`)

In [ ]:
settings = [
    (0.15, 6),
    (0.25, 6),
    (0.35, 6),
    (0.25, 12),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (eps, min_samples) in zip(axes.ravel(), settings):
    db = DBSCAN(eps=eps, min_samples=min_samples)
    y = db.fit_predict(X)
    noise_count = np.sum(y == -1)

    ax.scatter(X[y != -1, 0], X[y != -1, 1], c=y[y != -1], cmap="viridis", s=25, alpha=0.85, edgecolors="k", linewidth=0.2)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], c="red", marker="x", s=70, linewidth=1.8)
    ax.set_title(f"eps={eps}, min_samples={min_samples}, noise={noise_count}")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

## 5. Choose `eps` with k-distance plot

A common heuristic is to use the sorted distance to the `k`-th nearest neighbor, where `k = min_samples`.
Look for an "elbow" in the curve.

In [ ]:
min_samples = 6
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X)
distances, _ = neighbors_fit.kneighbors(X)

# Distance to the k-th nearest neighbor
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances, linewidth=2)
plt.title(f"k-distance Plot (k={min_samples})")
plt.xlabel("Points sorted by distance")
plt.ylabel("Distance to k-th nearest neighbor")
plt.grid(alpha=0.3)
plt.show()

## 6. Practice ideas

Try these in Colab:
1. Change dataset to `make_circles` and compare again.
2. Increase noise in `make_moons` and retune `eps`.
3. Add outliers manually and check DBSCAN robustness.
4. Compare runtime with K-Means on larger datasets.